# EGM 722 Project Notebook: Greenspace Analysis
---


## Introduction

Welcome to Greenspace Analysis project notebook! This will take you through the code required to perform analysis on Greenspace accessibility and availabilty in Northern Ireland. 

The code will consist of 2 main parts:
1. **Importing the necessary modules and loading the data**
2. **Carrying out the analysis**

The analysis is divided into 3 subsections, relating to a different kind of analysis technique demonstrated:
- Coverage Analysis
- Proximity Analysis
- Suitability Analysis

The data provided and used in this notebook consists of a number of andminstrative boundaries used for Northern Ireland, namely:
- Local Government Districts (LGD)
- District Electoral Areas (DEA)
- 2021 Census Data Zones (DZs)
- 2021 Census Super Data Zones (SDZs)
- Settlement Development Limits (SDLs)

Each analysis has been carried using the test data provided (e.g. counting greenspaces within Settlements rather than Data Zones or Super Data Zones), however
many of the functions however have been designed to allow a degree of flexibilty with the input data used, so feel free to experiment with the data provided and use the code to try different things.

---

## Part 1: Preparing Data

To start the project, we first need to import the necessary modules, followed by the data, and check that is in a consistent Co-ordinate Reference System (CRS).

In [ ]:
#Import the required modules
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from cartopy.feature import ShapelyFeature
import cartopy.crs as ccrs
import numpy as np
import rasterio as rio
import rasterstats
import rasterio.features
from rasterio.features import geometry_mask

In [ ]:
# Load and name the data layers
greenspace = gpd.read_file(os.path.abspath('data_files/Greenspace_Phase2_06052022.shp'))
lgd2012 = gpd.read_file(os.path.abspath('data_files/LGD2014.shp'))
dea2012 = gpd.read_file(os.path.abspath('data_files/dea2014.shp'))
dz2021 = gpd.read_file(os.path.abspath('data_files/DZ2021.shp'))
sdz2021 = gpd.read_file(os.path.abspath('data_files/SDZ2021.shp'))
settlements2015 = gpd.read_file(os.path.abspath('data_files/settlements-2015-above-500-threshold.shp'))


In [ ]:
# View the first 5 rows of the greenspace data to check if it loaded in correctly
greenspace

In [ ]:
# Finally, Check the crs of each layer
layer_crs = {'greenspace_crs': [greenspace.crs], 'lgd2012_crs': [lgd2012.crs], 'dea2012': [dea2012.crs], 'dz2021_crs': [dz2021.crs], 'sdz2021_crs': [sdz2021.crs], 'settlements2015_crs': [settlements2015.crs]} # Create a dictionary of layer/crs pairs
crs_table = pd.DataFrame(data=layer_crs) # convert to a dataframe to view
crs_table

Now that the data is correctly loaded and in the same CRS, we can begin the analysis.

## Part 2: Greenspace Analysis

This next part of the notebook will demonstrate some analysis using the greenspace data and boundaries provided.

Using the data, we can find:
1. What areas have the highest amounts of greenspace coverage at each level of geography.
2. What areas are closest to a greenspaces? How many greenspaces are within a particular distance of an area?
3. What areas have the largest amount of land available for potential greenspaces?


### Part 2.1. Calculating coverage

Upon inspecting the greenspace data, you may have noticed the original data contains a number of Multi Z Polygons. These are multipart polygons that are grouped together in one feature, and so when they are selected for calculating coverage in an area that overalps, the whole feature will be selected, returning a false representation of area/coverage. To get around this, the greenspace layer must be converted to single part polygons using the explode function.

In [ ]:
# Convert the greenspace polygons to single parts polygons and view the dataset
greenspace_sp=greenspace.explode(column=None)
greenspace_sp

Before the area of greenspace found within each SDZ, LGD, DEA etc. can be calculated, it would be useful to calculate the area of each polygon to a standard unit (Km^2).
This is where the first function will be defined, one that calculates the area of each polygon, in Km2 and adds it on as a new column to the table.

**Note**: This step is particularly important for the recently exploded greenspace layer, as the area of each single-part polygon created will need to be calculated in order to calculate their coverage.

In [ ]:
def area_calc(layer, col_name):
    """Caluclates the area of each polygon in the layer and returns a total.
    
    Parameters 
    layer : input polygon layer
    col_name : name of the area column

    Returns : sum_area_SQKM
        Prints the total area of the polygons in squared kilometers
    """

    layer[col_name] = layer['geometry'].area/1e6 # Create a new column calculating the area of each polygon in squared kilometers

    sum_area_SQKM = layer[col_name].sum() # Calculate the total area of the dataset
    print(f'The total area of this dataset is {sum_area_SQKM:.2f} kilometers squared.')

In [ ]:
#Use area_calc function on greenspace layer and display the results
area_calc(greenspace_sp, 'greenspace_area_SQKM')
greenspace_sp.head(6)

In [ ]:
# Add a area SQKM column to the remainder of the datasets and find their total area
area_calc(dz2021, 'dz_area')
area_calc(sdz2021, 'sdz_area')
area_calc(dea2012, 'dea_area')
area_calc(lgd2012, 'lgd_area')
area_calc(settlements2015, 'settlement_area')

Now we are ready to start analysing.

In [ ]:
def coverage_calc(layer, clipping_feature, orig_area):
    """
    Clips the selected greensapce layer to the selected dataset, creates a grouped gdf based on the individual features,
    which is then joined to the original selected dataset to calculate the % coverage

    Parameters: 
    layer - the selected layer or dataset to be clipped to
    clipping feature - the column name of the clipping dataset, must be a unique
    oirg_area - the calculated area column in the clipping dataset

    Returns:
    Output table - joined table of the original layer with information on its total coverage of greenspace
    """
    if layer[clipping_feature].is_unique == False:
        raise Exception('Clipping features must be unique.') # Raises an error if non unique features are used for the clipping
    
    clipped = [] # Create an empty list
    for selected_areas in layer[clipping_feature]: # iterate over each unique value in the clipping features
        tmp_clip = gpd.clip(greenspace_sp, layer[layer[clipping_feature] == selected_areas]) # clip the greenspace layer by each feature
        tmp_clip['greenspace_area_SQKM'] = tmp_clip['geometry'].area/1e6 # calculate the resulting area of the clipped greenspace polygons
        tmp_clip[clipping_feature] = selected_areas # set the name for each feature
    
        clipped.append(tmp_clip) # append the clipped GeoDataframe to the created list

    clipped_gdf = gpd.GeoDataFrame(pd.concat(clipped, ignore_index=True)) # Create a new geodataframe by combining the clipped geodataframes
    
    grouped_gdf = pd.DataFrame(index=layer[clipping_feature]) # Creates a grouped dataframe series that sums the total area of greenspace in the clipped layer
    grouped_gdf['greenspace_coverage_SQKM'] = clipped_gdf.groupby([clipping_feature])['greenspace_area_SQKM'].sum()

    output_table = pd.merge(layer, grouped_gdf, on = clipping_feature) # Joins the grouped data frame to the original dataset
    output_table['pc_coverage'] = output_table['greenspace_coverage_SQKM'] / output_table[orig_area] * 100 # calculates a new column to show the percentage of the area covered by greenspace

    return output_table

In [ ]:
dz_coverage = coverage_calc(dz2021, 'DZ2021_nm', 'dz_area') # Use the coverage_calc function to calculate the coverage of greenspace for each DZ
dz_coverage.sort_values(by=['pc_coverage'], ascending=False).head(10) # display the top 10 DZs in terms of percentage coverage

In [ ]:
sdz_coverage = coverage_calc(sdz2021, 'SDZ2021_nm', 'sdz_area') # Use the coverage_calc function to calculate the coverage of greenspace for each SDZ
sdz_coverage.sort_values(by=['pc_coverage'], ascending=False).head(10) # display the top 10 SDZs in terms of percentage coverage

In [ ]:
dea_coverage = coverage_calc(dea2012, 'DEA', 'dea_area') # Use the coverage_calc function to calculate the coverage of greenspace for each DEA
dea_coverage.sort_values(by=['pc_coverage'], ascending=False).head(10) # display the top 10 DEAs in terms of percentage coverage

In [ ]:
lgd_coverage = coverage_calc(lgd2012, 'LGDNAME', 'lgd_area') # Use the coverage_calc function to calculate the coverage of greenspace for each LGD
lgd_coverage.sort_values(by=['pc_coverage'], ascending=False).head(10) # display the top 10 LGDs in terms of greenspace coverage

Create a map showing the DEAs with the largest percentage cover of greenspaces

In [ ]:
def generate_handles(labels, colors, edge='k', alpha=1):
    """
    Generate matplotlib patch handles to create a legend of each of the features in the map.

    Parameters:
    
    labels - list(str)
        the text labels of the features to add to the legend

    colors - list(matplotlib color)
        the colors used for each of the features included in the map.

    edge - matplotlib color (default: 'k')
        the color to use for the edge of the legend patches.

    alpha - float (default: 1.0)
        the alpha value to use for the legend patches.

    Returns: 
    
    handles - list(matplotlib.patches.Rectangle)
        the list of legend patches to pass to ax.legend()
    """
    lc = len(colors)  # get the length of the color list
    handles = [] # create an empty list
    for ii in range(len(labels)): # for each label and color pair that we're given, make an empty box to pass to our legend
        handles.append(mpatches.Rectangle((0, 0), 1, 1, facecolor=colors[ii % lc], edgecolor=edge, alpha=alpha))
    return handles

# adapted this question: https://stackoverflow.com/q/32333870
# answered by SO user Siyh: https://stackoverflow.com/a/35705477
def scale_bar(ax, length=20, location=(0.92, 0.95)):
    
    """
    Create a scale bar in a cartopy GeoAxes.

    Parameters

    ax - cartopy.mpl.geoaxes.GeoAxes
        the cartopy GeoAxes to add the scalebar to.

    length - int, float (default 20)
        the length of the scalebar, in km

    location - tuple(float, float) (default (0.92, 0.95))
        the location of the center right corner of the scalebar, in fractions of the axis.

    Returns:
    ax - cartopy.mpl.geoaxes.GeoAxes
        the cartopy GeoAxes object

    """
    x0, x1, y0, y1 = ax.get_extent() # get the current extent of the axis
    sbx = x0 + (x1 - x0) * location[0] # get the right x coordinate of the scale bar
    sby = y0 + (y1 - y0) * location[1] # get the right y coordinate of the scale bar

    ax.plot([sbx, sbx-length*1000], [sby, sby], color='k', linewidth=4, transform=ax.projection) # plot a thick black line
    ax.plot([sbx-(length/2)*1000, sbx-length*1000], [sby, sby], color='w', linewidth=2, transform=ax.projection) # plot a white line from 0 to halfway

    ax.text(sbx, sby-(length/4)*1000, f"{length} km", ha='center', transform=ax.projection, fontsize=6) # add a label at the right side
    ax.text(sbx-(length/2)*1000, sby-(length/4)*1000, f"{int(length/2)} km", ha='center', transform=ax.projection, fontsize=6) # add a label in the center
    ax.text(sbx-length*1000, sby-(length/4)*1000, '0 km', ha='center', transform=ax.projection, fontsize=6) # add a label at the left side

    return ax

In [ ]:
ni_utm = ccrs.UTM(29) # Create a Universal Transverse Mercator reference system to transform the data.
ccrs.CRS(greenspace.crs) # Create a cartopy CRS representation of the CRS associated with the greenspace dataset

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=ni_utm)

# Add DEAs to the map 
DEAs = ShapelyFeature(dea2012['geometry'], ni_utm, edgecolor = 'grey', facecolor = 'lightgray', linewidth = 1)
ax.add_feature(DEAs)   

# Add LGDs to the map
LGDs = ShapelyFeature(lgd2012['geometry'], ni_utm, edgecolor = 'k', facecolor = 'w', linewidth = 2, alpha=0.5)
ax.add_feature(LGDs)

#Add Greenspaces to the map
Greenspaces = ShapelyFeature(greenspace['geometry'], ni_utm, edgecolor = 'g', facecolor ='g', alpha=0.85)
ax.add_feature(Greenspaces)    
                                      
xmin, ymin, xmax, ymax = greenspace.total_bounds # using the boundary of the shapefile features, zoom the map to our area of interest
ax.set_extent([xmin-5000, xmax+5000, ymin-5000, ymax+5000], crs=ni_utm) # we re-order the coordinates to work with set_extent.

# Generate handles for each layer
lgd_handles = generate_handles(['LGDNAME'], ['w']) 
dea_handles = generate_handles(['DEA'], ['lightgray'])
greenspace_handles = generate_handles(['Name'], ['g'])

# Create the legend using the handles
handles = lgd_handles + dea_handles + greenspace_handles 
labels = ['Local Government Districts', 'District Electoral Areas', 'Greenspaces']

leg = ax.legend(handles, labels, title='Legend', title_fontsize=12, fontsize=10, loc='upper left', frameon=True, framealpha=1) # Add legend to the map

scale_bar(ax) # Add a scale bar to the map

fig.savefig('output_files/map.png', bbox_inches='tight', dpi=300) # Save figure to output files folder

### Part 2.2. Finding Distances

The next part of the analysis will demonstrate how python can be used to calculate distances between greenspaces and the data zones (DZ). This will consist of: 

a) A buffer analysis, creating a function that allows you to easily calculate the number of greenspaces within a specified distance of your chosen location.
b) An interactive map, that shows the distance to the nearest greenspace for each DZ in the country.

In [ ]:
def buffer_analysis(layer, name_col, dist, dist_col): # Define a function to count the number of greenspaces within a specified distance of features in a layer

    '''
    Performs a distance within spatial join to count the number of greenspaces within specified distance of a chosen features,
    then returns a the a merged copy of the original dataframe showing counts in a new column.

    Parameters:

    layer - input layer to count from
    name_col - column name containing the name of each chosen feature
    dist - buffer distance, in m
    dist_col - name of the returned column

    Returns:

    dist_table - a copy of the original layer, merged to include greenspace counts for specified distance

    '''

    within_dist= gpd.sjoin(layer, greenspace[['geometry']], how='inner', predicate='dwithin', distance=(dist)) # spatial join using the distance within function, to join the number of greenspaces to each feature
   
    # Create a grouped geodataframe counting the number of greenspace
    num_gspace = within_dist.groupby(name_col)
    num_gspace_df = pd.DataFrame(index=layer[name_col])
    num_gspace_df[dist_col] = num_gspace[name_col].count()

    counts = num_gspace_df.fillna(0) # fill any NaN values with 0
    dist_table = layer.merge(counts, left_on=[name_col], right_on=[name_col]) # merge to a new copy of the original layer

    return dist_table


In [ ]:
# Use buffer_analysis to find the number of greenspaces within 300m, 2km, 5km and 10km of settlement areas
gspace_300m=buffer_analysis(settlements2015, 'Name', 300, 'gSpaces_within_300m')
gspace_2km=buffer_analysis(settlements2015, 'Name', 2000, 'gSpaces_within_2km')
gspace_5km=buffer_analysis(settlements2015, 'Name', 5000, 'gSpaces_within_5km')
gspace_10km=buffer_analysis(settlements2015, 'Name', 10000, 'gSpaces_within_10km')

# Merge each result to the main settlement table, using the settlement codes
settlements_dists = pd.merge(settlements2015, gspace_300m[['Name', 'gSpaces_within_300m']], on= 'Name', how= 'left')
settlements_dists = pd.merge(settlements_dists, gspace_2km[['Name', 'gSpaces_within_2km']], on= 'Name', how= 'left')
settlements_dists= pd.merge(settlements_dists, gspace_5km[['Name', 'gSpaces_within_5km']], on= 'Name', how= 'left')
settlements_dists = pd.merge(settlements_dists, gspace_10km[['Name', 'gSpaces_within_10km']], on= 'Name', how= 'left')

settlements_dists

While this information is useful by itself, it could be summarised by local government district to show the typical number of greenspaces available for a settlement. 
To show this, the settlement layer must be joined to the LGD layer. Since settlement boundaries may overlap, deriving representative points will ensure a cleaner join.

In [ ]:
settlement_pt = settlements_dists.copy() # Make a copy of each the dataset
lgd_copy = lgd2012.copy()
settlement_pt['geometry']=settlement_pt['geometry'].representative_point() # Overwrite the geometry to representative points

contains = gpd.sjoin(lgd_copy, settlement_pt, how='inner', predicate='contains') # Peform a spatial join to find what lgd each settlement belongs to

columns = ['gSpaces_within_300m', 'gSpaces_within_2km', 'gSpaces_within_5km', 'gSpaces_within_10km'] # columns to summarize

lgd_summary_gSpace = contains.groupby(['LGDNAME'], as_index=False)[columns].median() # Create a grouped geodataframe, finding the median of each column
lgd_summary_gSpace['num_settlements'] = contains.groupby(['LGDNAME'])['Code'].nunique().values # include a column counting the number of settlements

lgd_summary_gSpace

In [ ]:
#write this to a csv file and save in output folder
lgd_summary_gSpace.to_csv('output_files/lgd_summary_gSpace.csv')

Another way to estimate each areas proximity to a greenspace is by calculating each area's distance to a greenspace. The DZ layer will be used to demonstrate this, and create an interactive map showing how far each area is from a greenspace.

In [ ]:
for ind, row in dz2021.iterrows(): # Iterrate over each row in dz
    pt = row['geometry'].centroid # generate a centroid as pt
    distances = greenspace.distance(pt) # calculate the distance from each centroid to each greenspace

    min_ind = distances.argmin() # Get the index of the mininum value
    min_dist = distances.min() # get the minimum distace

    dz2021.loc[ind, 'Nearest_gSpace'] = greenspace.loc[min_ind].Name # Report the name of the nearest greenspace
    dz2021.loc[ind, 'Distance_km'] = min_dist/1000 # create a new column calculating the minimum distance in km

dz2021.Distance_km = dz2021.Distance_km.round(2) # Round the distance column to 2.dp

The next part of the code will count the number of greenspaces in each dz, using a spatial join:

In [ ]:
joined = gpd.sjoin(dz2021, greenspace, how='inner', lsuffix='left', rsuffix='right') # Create a gdf using a spatial join to list the dz each greenspace polygon is found in

num_gspace = joined.groupby('DZ2021_nm') # Group by DZ name
num_space_df = pd.DataFrame(index=dz2021['DZ2021_nm']) # Create a grouped dataframe that counts the number of occurrences (greenspaces) in each dz as a column "Num_gSpaces"
num_space_df['Num_gSpaces'] = num_gspace['DZ2021_nm'].count()

to_join = num_space_df.fillna(0) # Set any NaN values to equal to 0 to ensure the mergre will work

merged = dz2021.merge(to_join, left_on='DZ2021_nm', right_on='DZ2021_nm') # Merge the dataframe to the original dz2021 layer, using the dz names
merged # show the result

In [ ]:
nearest_gspace_desc = merged.describe()# Obtain descriptive statistics and export result to csv.
nearest_gspace_desc.to_csv('output_files/nearest_gspace.csv')

In [ ]:
m = merged.explore('Distance_km', # show the Distance column
                   cmap='Blues_r', # Set the colourmap to blues
                   legends_kwds={'caption': 'Distance to nearest greenspace in km'} # add the legents
                                 )

greenspace.explore('Category', # Plot the greenspace layer showing the different categories
                   m=m, # add to existing map
                   cmap = 'Set2', # set colourmap to set 2
                   popup= True, # enable pop ups
                   legend_kwds= {'caption': 'Greenspace Category'}#add legend
                  )
                  

In [ ]:
m.save('output_files/greenspace_imap.html') # Save as an html file

### Part 2.3. Potential Greenspace

This next part of the analysis will involve calculating the area of potential greenspace for each settlement. It will use a landcover layer of 100m resolution to help identify potential areas of greenspace, and exclude any areas that area already designated greenspaces.

First, the raster layer must be prepared by loading the layer and defining the land cover class names and creating a dict object of key/value pairs

In [ ]:
# Define the landcover class names in a list
names = ['Broadleaf woodland', 'Coniferous woodland', 'Arable', 'Improved grassland', 'Semi-natural grassland',
         'Mountain, heath, bog', 'Saltwater', 'Freshwater', 'Coastal', 'Built-up areas and gardens']

values = range(1, 11) # Get numbers from 1-10, corresponding to the landcover values

landcover_names = dict(zip(values, names)) # Create a landcover dict of value/name pairs

#load the landcover raster and read the data
with rio.open('data_files/LCM2015_Aggregate_100m.tif') as dataset:
    xmin, ymin, xmax, ymax = dataset.bounds
    crs = dataset.crs
    landcover2015 = dataset.read(1)
    affine_tfm = dataset.transform

print(dataset.crs) # Check the crs

With the raster loaded and classes defined, it's needs reprojected to match the other shapefiles used in the analysis : EPSG 29902

In [ ]:
#Reproject the Raster to EPSG 29902
dst_crs = 'epsg:29902' # Define the destination crs (EPSG: 29902)

with rio.open('data_files/LCM2015_Aggregate_100m.tif') as src: # Set the original raster as the source dataset
    transform, width, height = rio.warp.calculate_default_transform( # find the new transform, width and height attribute values for the reprojected raster
        src.crs, dst_crs, src.width, src.height, *src.bounds)
    
    kwargs = src.meta.copy() # copy the meta attribute from the source dataset
    
    kwargs.update({ # Changes the attributes of the dict
    'crs': dst_crs,
    'transform': transform,
    'width': width,
    'height': height
    })

    with rio.open('data_files/LCM2015_Aggregate_100m_reproj.tif', 'w', **kwargs) as dst: # Writes the reprojected raster as a new file 
        for ind in range(1, src.count +1): # Reproject each band from the source dataset to the reporjected raster, using a nearest-neighbour resampling
            rio.warp.reproject(
                source=rio.band(src, ind),
                destination=rio.band(dst, ind),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=rio.warp.Resampling.nearest
            )

In [ ]:
with rio.open('data_files/LCM2015_Aggregate_100m_reproj.tif') as dataset: # Load and read the newly reprojected raster as dataset
    xmin, ymin, xmax, ymax = dataset.bounds
    crs = dataset.crs
    landcover2015_reproj = dataset.read(1)
    affine_tfm = dataset.transform

    print(dataset.crs) # check the crs to verify the reprojection worked

Now that the classes have defined and labelled, and the raster reprojected, a reclassified copy of the dataset will be made to find only the desired classes (areas suitable for greenspace): broadleaf and coniferous woodlands and semi-natural grassland.

In [ ]:
landcover_rc = landcover2015_reproj.copy() # Make a copy of the reprojected raster to reclassify

landcover_rc[np.where((landcover_rc == 1) | (landcover_rc == 2) | (landcover_rc == 5))] = 1 # Set suitable classes equal to 1
landcover_rc[np.where(
    (landcover_rc == 3) |
    (landcover_rc == 4) |
    (landcover_rc == 6) |
    (landcover_rc == 7) |
    (landcover_rc == 8) |
    (landcover_rc == 9) |
    (landcover_rc == 10) |
    (landcover_rc == 11))] = 2 # Set unsuitable classes equal to 2

class_names = { # Define the new class names
    1: "Suitable",
    2: "Unsuitable"
}

When looking for potential greenspaces, areas that are already allocated greenspaces should be excluded. This will be done by masking the greenspace layer.

In [ ]:
# convert the greenspace layer to raster
green_mask = geometry_mask( 
    geometries=greenspace.geometry, # Use geometries from greenspace layer
    transform=affine_tfm, # geotransform the new raster
    invert = True, # invert, so values within polygons will = true
    out_shape=landcover_rc.shape)

landcover_rc[green_mask] = 2 # select cells within masked cells (where cells are true) and assign the value of 0

With the areas of current greenspace excluded, it is now possible to find potential greenspace each settlement has using zonal statistics of a polygon. This will be done to calculate the potential for areas within 0.3, 1, 2, and 5 km, of a chosen polygon layer.
In order to repeat this process for the different buffer zones, a function will be defined to calculate the potential greenspace for a selected layer, in kilometers squared:

In [ ]:
# Write this as a function

def potential_gp_calc (input_layer, name_col, buffer_dist, suitable_area_col, unsuitable_area_col):
    '''
    Count the number of pixels suitable/unsuitable suitable for greenspace, within a distance of chosen features

    Parameters:
    input_layer: the layer to used to count the potential greenspaces from
    name_col: column containing names or IDs of the chosen features e.g. settlement names or ward codes
    buffer_dist: the selected search distance, in metres
    suitable_area_col: output column name for storing the calculated suitable area, in kilometers squared
    unsuitable_area_col: output column name for storing the calculated unsuitable area, in kilometers squared

    returns:
    buffered: output copy GeoDataFrame of original layer, with calculated areas and pixel counts attached

    '''
    
    buffered = input_layer.copy()
    buffered['geometry'] = buffered.geometry.buffer(buffer_dist)

    potential_gs = rasterstats.zonal_stats(buffered, # Polygons to use
                                           landcover_rc, # raster to use (land cover)
                                           affine = affine_tfm, # geotransform the raster
                                           categorical = True, # verifies data is categorical 
                                           category_map = class_names, # categories to use
                                           nodata = 0 # fill no data values
                                          )
    names = buffered[name_col]

    layer_dict = dict(zip(names, potential_gs))

    for ind, row in buffered.iterrows(): #use iterrows to iterate oer the rows of the table
        layer_data = layer_dict[row[name_col]] # get the suitability count for this settlement
        for name in class_names.values(): # iterate over each of the suitability classes
            if name in layer_data.keys(): # check that name is a key of settlment_data
                buffered.loc[ind, name] = layer_data[name] # add the suitability count to a new colomn
            else:
                buffered.loc[ind, name] = 0 #if name is not present, value should be 0

    #Add columns showing the area of suitable and unsuitable
    buffered[suitable_area_col] = buffered['Suitable']*0.01
    buffered[unsuitable_area_col] = buffered['Unsuitable']*0.01

    return buffered


Now this function will be used to find the area of suitable/unsuitable greenspaces within 300m, 1km, 2km and 5km, of each settlement area

In [ ]:
#Use this function to find areas within 300m, 1, 2, and 5 km
potential_gp_300m = potential_gp_calc(settlements2015, 'Name', 300, 'Suitable_area_300m', 'Unsuitable_area_300m')
potential_gp_2km = potential_gp_calc(settlements2015, 'Name', 2000, 'Suitable_area_2km', 'Unsuitable_area_2km')
potential_gp_5km = potential_gp_calc(settlements2015, 'Name', 5000, 'Suitable_area_5km', 'Unsuitable_area_5km')
potential_gp_10km = potential_gp_calc(settlements2015, 'Name', 10000, 'Suitable_area_10km', 'Unsuitable_area_10km')

#Merge each result to the main settlement table, on the settlement codes
settlements_merged = pd.merge(settlements2015, potential_gp_300m[['Code', 'Suitable_area_300m', 'Unsuitable_area_300m']], on= 'Code', how= 'left')
settlements_merged = pd.merge(settlements_merged, potential_gp_2km[['Code', 'Suitable_area_2km', 'Unsuitable_area_2km']], on= 'Code', how= 'left')
settlements_merged = pd.merge(settlements_merged, potential_gp_5km[['Code', 'Suitable_area_5km', 'Unsuitable_area_5km']], on= 'Code', how= 'left')
settlements_merged = pd.merge(settlements_merged, potential_gp_10km[['Code', 'Suitable_area_10km', 'Unsuitable_area_10km']], on= 'Code', how= 'left')

In [ ]:
settlements_merged.sort_values(by=['Suitable_area_10km'], ascending=False) # view the merged dataset and show the area with the highest amount of potential greenspace

And finally to finsh the analysis, summarise the table information above by deriving descriptive statistics for greenspace availability

In [ ]:
settlements_pgp_summary = settlements_merged.describe() #  derive descriptive statistics for potential greenspace availability
settlements_pgp_summary.to_csv('output_files/settlments_pgp_summary.csv') # export to csv

---

**This is the end of the analysis**

---